In [1]:
# Chap 16. 다양한 댓글 모으기
# 테스트 기사 URL :
# https://www.youtube.com/shorts/UkU_P-jfBtM

#Step 1. 필요한 모듈과 라이브러리를 로딩합니다.

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import numpy
import pandas as pd
import random
import os
import re

#Step 2. 사용자에게 검색어 키워드를 입력 받고 저장할 폴더와 파일명을 설정합니다.
print("=" *80)
print(" Chap 16.쇼츠 댓글 정보 수집하기")
print("=" *80)
print("\n")

query_txt = '쇼츠댓글'
# query_url = input('1.댓글을 크롤링할 쇼츠의 URL을 입력하세요: ')
cnt = int(input('2.크롤링 할 건수는 몇건입니까?(10건단위로 입력): '))
page_cnt = math.ceil(cnt)

f_dir = input("3.파일을 저장할 폴더명만 쓰세요(예:c:\\py_temp\\):")
if f_dir=='' :
    f_dir='c:\\py_temp\\결과 추출 - 리뷰, 댓글\\'

if cnt > 10 :
      print("요청 건수가 많아서 시간이 제법 소요되오니 잠시만 기다려 주세요~~")
else :
      print("요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~")

# 저장될 파일위치와 이름을 지정합니다
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s+'-'+query_txt)
os.chdir(f_dir+s+'-'+query_txt)

ff_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xls'

#Step 3. 크롬 드라이버를 사용해서 웹 브라우저를 실행합니다.

s_time = time.time( )

s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get("https://www.youtube.com/shorts/UkU_P-jfBtM")
driver.maximize_window()
time.sleep(5)

#Step 4. 현재 총 리뷰 건수를 확인하여 사용자의 요청건수와 비교 후 동기화합니다
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

result= soup.find('button-view-model','ytSpecButtonViewModelHost ytwReelActionBarViewModelHostDesktopActionButton').find('span','yt-core-attributed-string')		#결과 뽑는거. 이게 중요!*!*!*
result2 = result.get_text()		#리뷰 개수 뽑는 것

print("=" *80)
result3 = result2.replace(",","")		#1,000건이 넘으면 중간에 ,가 있어서 제거하는거.
result4 = re.search("\d+",result3)
search_cnt = int(result4.group())

if cnt > search_cnt :
    cnt = search_cnt

print("전체 검색 결과 건수 :",search_cnt,"건")
print("실제 최종 출력 건수",cnt)
# print("실체 출력될 최종 페이지수" , page_cnt)     29번줄과 함께.


<>:71: SyntaxWarning: invalid escape sequence '\d'
<>:71: SyntaxWarning: invalid escape sequence '\d'
C:\Users\itwill\AppData\Local\Temp\ipykernel_9760\2808589939.py:71: SyntaxWarning: invalid escape sequence '\d'
  result4 = re.search("\d+",result3)


 Chap 16.쇼츠 댓글 정보 수집하기


요청 건수가 많아서 시간이 제법 소요되오니 잠시만 기다려 주세요~~
전체 검색 결과 건수 : 22 건
실제 최종 출력 건수 22


In [2]:
# Step 5. 사용자가 요청한 건수가 많을 경우 리뷰 더보기 버튼을 클릭합니다
# 최초 10건 수집후 댓글 더보기 버튼 클릭
# 아래 버튼을 눌러 첫 화면에 총 20건의 댓글이 나오게 만듦
driver.find_element(By.XPATH,'//*[@id="button-bar"]/reel-action-bar-view-model/button-view-model[1]').click()		#댓글 더보기(5->2
time.sleep(3)

#Step 6. 리뷰와 점수 등 내용 수집
y_URL2=[] # 동영상 URL
reviewer2=[] # 댓글작성자명
review_d2=[] #댓글 작성일자
review2=[] #리뷰내용
like2=[] #좋아요횟수
count = 0
prev_count = -1  # ✅ 이전 루프에서 수집된 댓글 수 추적용

for a in range(1, page_cnt+1):

    print('이제 리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~')
    f = open(ff_name, 'a', encoding='UTF-8')

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    y_URL = driver.current_url
    print(f"\n[현재 수집 중인 동영상 URL: {y_URL}]")

    reple_result = soup.find('div', id='contents').find_all('ytd-comment-thread-renderer')

    for li in reple_result:

        # ✅ 대댓글 제외: 기본 댓글 영역이 없는 항목은 건너뜀
        main_comment = li.find('ytd-comment-view-model')
        if not main_comment:
            continue

        count += 1
        print("\n")
        print("총 %s건 중 %s번째 댓글 수집 중입니다 =========" % (cnt, count))

        print("1.동영상 URL:", y_URL)
        f.write("\n")
        f.write("총 %s 건 중 %s 번째 리뷰 데이터를 수집합니다====\n" % (cnt, count))
        f.write("1.동영상 URL: " + y_URL + "\n")
        y_URL2.append(y_URL)
        time.sleep(1)

        reviewer = li.find('div', id='header-author').find('a', 'yt-simple-endpoint style-scope ytd-comment-view-model').get_text()
        print("2.댓글작성자명:", reviewer)
        f.write("2.댓글작성자명: " + reviewer.replace("\n", "").strip() + "\n")
        reviewer2.append(reviewer)
        time.sleep(2)

        review_d = li.find('div', id='header-author').find('span', id='published-time-text').get_text()
        print("3.댓글작성일자:", review_d)
        f.write("3.댓글작성일자: " + review_d.replace("\n", "").strip() + "\n")
        review_d2.append(review_d)
        time.sleep(1)

        review = li.find('div', id='content').find('span', 'yt-core-attributed-string yt-core-attributed-string--white-space-pre-wrap').get_text()
        print("4.리뷰내용:", review)
        f.write("4.리뷰내용: " + review + "\n")
        review2.append(review)
        time.sleep(2)

        try:
            like_node = li.select_one('like-button-view-model span[role="text"]')
            if like_node:
                like = like_node.get_text(strip=True)
            else:
                backup_node = li.select_one('#vote-count-middle')
                like = backup_node.get_text(strip=True) if backup_node else '0'
            if not like:
                like = '0'
        except Exception as e:
            like = '0'

        print("5.좋아요횟수:", like)
        f.write("5.좋아요횟수:" + like + "\n")
        like2.append(like)

        # ✅ 요청 건수 도달 시 내부 루프 즉시 중단
        if count >= cnt:
            print(f"\n✅ 요청 건수 {cnt}건 수집 완료!")
            break

    time.sleep(0.2)

    # ✅ 새 댓글이 없으면 (기본 댓글 소진) 외부 루프 중단
    if count == prev_count:
        print(f"\n✅ 더 이상 수집할 기본 댓글이 없습니다. (총 {count}건 수집 완료)")
        break

    prev_count = count

    # ✅ 외부 루프도 중단
    if count >= cnt:
        break

이제 리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~

[현재 수집 중인 동영상 URL: https://www.youtube.com/shorts/UkU_P-jfBtM]


총 22건 중 1번째 댓글 수집 중입니다 =========
1.동영상 URL: https://www.youtube.com/shorts/UkU_P-jfBtM
2.댓글작성자명: 
 @catch_speed 

3.댓글작성일자: 

            5개월 전
          

4.리뷰내용: 올리브영 알바생이 추천하는 꿀템 5가지

1. 짱구 향수
짱구덕후라면 꼭 샤야된다. 짱구도 귀엽고 파우치 키링까지 주는데 포근한 코튼향이라 내 살냄새같아서 짱조음

2. 콜레올로지 컷팅 젤리 

아이돌 다이어트 필수템인 컷팅 젤리인데 간식 땡길 때 쭉쭉 짜먹으면 좋아요! 체지방 컷팅에 식후혈당까지 다잡아줘서 가방에 하나씩 넣고다니기 좋음! 

3. 도로시와 퍼스널 컬러 컷 누드브라 
너무너무 좋아서 여행갈때도 챙겨갔는데 퍼스널 컬러 컷 누드브라 이름처럼 컬러가 여러개라 
내 피부톤에 찰떡인 컬러로 고르면 티 1도 안남!! 촉감도 부드럽고 얇아서 역대급으로 자연스럽고 라인도이쁨♡

4. 메디큐브 핑크콜라겐 캡슐크림
저 이제 피부과 안갑니다. 분홍색 캡슐을 톡톡 터트리며 발라주면 담날 트러블 다들어가요

5. 레인보우 치즈 베이글칩
치즈맛 제대로나고 바삭바삭한게 진짜 맛있어서 입심심할때 다이어트 과자로 좋아요
5.좋아요횟수: 31


총 22건 중 2번째 댓글 수집 중입니다 =========
1.동영상 URL: https://www.youtube.com/shorts/UkU_P-jfBtM
2.댓글작성자명: 
 @뿡뿡이-d3q 

3.댓글작성일자: 

            1개월 전
          

4.리뷰내용: 저거 캡슐크림 달바꺼도 진짜 좋아요!! 비타민이 여러모로 최고에요.. 갠적으로 스킨케어할 때 화장품도 비타민 성분만 쓰는데 바이오힐보 비타민토너, 비타치올 비타민앰플, 달바 비타크림으

In [3]:
#Step 7. xls 형태와 csv 형태로 저장하기
news_reple = pd.DataFrame()
news_reple['동영상 URL']=pd.Series(y_URL2)
news_reple['댓글작성자명']=pd.Series(reviewer2)
news_reple['댓글 작성일자']=pd.Series(review_d2)
news_reple['리뷰내용']=pd.Series(review2)
news_reple['좋아요횟수']=pd.Series(like2)

# csv 형태로 저장하기
news_reple.to_csv(fc_name,encoding="utf-8-sig",index=True)

# 엑셀 형태로 저장하기
news_reple.to_excel(fx_name ,index=True , engine='openpyxl')

# Step 8. 요약 정보 출력하기
e_time = time.time( )
t_time = e_time - s_time

print("\n")
print("=" *120)
print("1.요청된 총 %s 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 %s 건입니다" %(cnt,count))
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xls 파일명 : %s " %fx_name)
print("=" *120)
driver.close()



1.요청된 총 22 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 22 건입니다
2.총 소요시간은 275.6 초 입니다 
3.파일 저장 완료: txt 파일명 : c:\py_temp\결과 추출 - 리뷰, 댓글\2026-03-11-17-20-00-쇼츠댓글\2026-03-11-17-20-00-쇼츠댓글.txt 
4.파일 저장 완료: csv 파일명 : c:\py_temp\결과 추출 - 리뷰, 댓글\2026-03-11-17-20-00-쇼츠댓글\2026-03-11-17-20-00-쇼츠댓글.csv 
5.파일 저장 완료: xls 파일명 : c:\py_temp\결과 추출 - 리뷰, 댓글\2026-03-11-17-20-00-쇼츠댓글\2026-03-11-17-20-00-쇼츠댓글.xls 
